In [ ]:
%pip install -q -U ultralytics pyyaml


# YOLO26 Vehicle Detection 

# Label Format
`class_id x_center y_center width height` (YOLO normalized).

In [ ]:
from pathlib import Path

SOURCE_ROOT = Path('/kaggle/input/datasets/sakshamjn/vehicle-detection-8-classes-object-detection/train')
SOURCE_IMAGES = SOURCE_ROOT / 'images'
SOURCE_LABELS = SOURCE_ROOT / 'labels'
WORKDIR = Path('/kaggle/working/dataset')

image_files = sorted([p for p in SOURCE_IMAGES.glob('*.jpg')])
label_files = sorted([p for p in SOURCE_LABELS.glob('*.txt') if p.name != 'classes.txt'])

quick_overview = {
    'source_images_exists': SOURCE_IMAGES.exists(),
    'source_labels_exists': SOURCE_LABELS.exists(),
    'image_count': len(image_files),
    'label_count': len(label_files),
    'workdir': str(WORKDIR),
}
quick_overview


In [ ]:
import shutil
from sklearn.model_selection import train_test_split

pairs = [(img, SOURCE_LABELS / f'{img.stem}.txt') for img in image_files]
assert pairs, 'Dataset kosong'
assert all(lbl.exists() for _, lbl in pairs), 'Ada label yang belum tersedia'

train_pairs, temp_pairs = train_test_split(pairs, test_size=0.2, random_state=42, shuffle=True)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=42, shuffle=True)

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

for split in ('train', 'val', 'test'):
    (WORKDIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (WORKDIR / split / 'labels').mkdir(parents=True, exist_ok=True)

for split_name, split_pairs in (('train', train_pairs), ('val', val_pairs), ('test', test_pairs)):
    for img_path, lbl_path in split_pairs:
        shutil.copy2(img_path, WORKDIR / split_name / 'images' / img_path.name)
        shutil.copy2(lbl_path, WORKDIR / split_name / 'labels' / lbl_path.name)

split_report = {
    'train': len(train_pairs),
    'val': len(val_pairs),
    'test': len(test_pairs),
}
split_report


In [ ]:
import yaml

classes_path = SOURCE_LABELS / 'classes.txt'
class_names = [line.strip() for line in classes_path.read_text().splitlines() if line.strip()]
assert class_names, 'classes.txt kosong atau tidak valid'

invalid_labels = []
for split in ['train', 'val', 'test']:
    split_label_dir = WORKDIR / split / 'labels'
    for label_file in split_label_dir.glob('*.txt'):
        rows = [row.strip() for row in label_file.read_text().splitlines() if row.strip()]
        for row in rows:
            parts = row.split()
            if len(parts) != 5:
                invalid_labels.append((label_file.name, 'format'))
                continue
            cls_id = int(float(parts[0]))
            coords = [float(v) for v in parts[1:]]
            if cls_id < 0 or cls_id >= len(class_names):
                invalid_labels.append((label_file.name, 'class_id'))
            if any(v < 0 or v > 1 for v in coords):
                invalid_labels.append((label_file.name, 'bbox_range'))

assert not invalid_labels, f'Invalid label rows found: {len(invalid_labels)}'

data_yaml = {
    'path': str(WORKDIR),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(class_names),
    'names': class_names,
}

data_yaml_path = WORKDIR / 'data.yaml'
with data_yaml_path.open('w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

data_yaml


In [ ]:
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

MODEL_WEIGHTS = 'yolo26n.pt'
model = YOLO(MODEL_WEIGHTS)

gpu_count = torch.cuda.device_count()
device = [0, 1] if gpu_count >= 2 else (0 if gpu_count == 1 else 'cpu')
batch_size = 32 if gpu_count >= 2 else 16

train_results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=batch_size,
    seed=42,
    patience=20,
    project='/kaggle/working/runs',
    name='detect_yolo26_vehicle',
    exist_ok=True,
    verbose=False,
    plots=True,
    save=True,
    val=True,
    device=device,
)

run_dir = Path('/kaggle/working/runs/detect_yolo26_vehicle/weights')
best_src = run_dir / 'best.pt'
last_src = run_dir / 'last.pt'

models_dir = Path('/kaggle/working/models')
models_dir.mkdir(parents=True, exist_ok=True)

best_dst = models_dir / 'yolo26_vehicle_best.pt'
last_dst = models_dir / 'yolo26_vehicle_last.pt'

if best_src.exists():
    shutil.copy2(best_src, best_dst)
if last_src.exists():
    shutil.copy2(last_src, last_dst)

runtime_train_info = {
    'gpu_count': gpu_count,
    'device_used': device,
    'batch_size': batch_size,
    'saved_models': {
        'best': str(best_dst) if best_dst.exists() else None,
        'last': str(last_dst) if last_dst.exists() else None,
    },
}

{
    'runtime': runtime_train_info,
    'train': train_results.results_dict if hasattr(train_results, 'results_dict') else train_results,
}


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

best_weights = '/kaggle/working/runs/detect_yolo26_vehicle/weights/best.pt'
model = YOLO(best_weights)

test_metrics = model.val(data=str(data_yaml_path), split='test', plots=True, verbose=False)

test_image_dir = WORKDIR / 'test' / 'images'
test_images = sorted([p for p in test_image_dir.glob('*.jpg')])
sample_count = min(5, len(test_images))
sample_images = random.sample(test_images, sample_count) if sample_count > 0 else []

if sample_count > 0:
    fig, axes = plt.subplots(sample_count, 1, figsize=(10, sample_count * 5))
    if sample_count == 1:
        axes = [axes]

    for i, image_path in enumerate(sample_images):
        img = cv2.imread(str(image_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = model(str(image_path), verbose=False)

        axes[i].imshow(img_rgb)
        axes[i].axis('off')

        for pred in results[0].boxes:
            x1, y1, x2, y2 = pred.xyxy[0].cpu().numpy()
            conf = float(pred.conf.cpu().numpy())
            class_id = int(pred.cls.cpu().numpy())
            label = model.names[class_id]
            axes[i].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='red', linewidth=2))
            axes[i].text(
                x1,
                y1,
                f'{label}: {conf:.2f}',
                color='white',
                fontsize=12,
                bbox=dict(facecolor='red', alpha=0.5),
            )

    plt.tight_layout()
    plt.show()

test_metrics.results_dict if hasattr(test_metrics, 'results_dict') else test_metrics
